In [0]:
rawdf1=spark.read.csv("/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified", header=False, inferSchema=True).toDF("id","fname","lname","age","profession")
rawdf1.show(200,False)
display(rawdf1.take(20))
display(rawdf1.sample(.1))
rawdf1.printSchema()
rawdf1.count()


In [0]:
print(rawdf1.columns)
print(rawdf1.dtypes)
for i in rawdf1.dtypes:
    if i[1]=='string':
        print(i[0])
print(rawdf1.schema)

print("actual count",rawdf1.count())
print("distinct count",rawdf1.distinct().count())
print("distinct count",rawdf1.select("fname").distinct().count(")


In [0]:
print("actual count",rawdf1.count())
print("de-duplicated record all columns count",rawdf1.distinct().count())
print("de-duplicated record all columns count",rawdf1.dropDuplicates().count())
print("de-duplicated record given col",rawdf1.dropDuplicates(['id']).count())
display(rawdf1.describe())
display(rawdf1.summary())

In [0]:
from pyspark.sql import SparkSession
spark= SparkSession.builder.appName("BB2-ETL Pipeline").getOrCreate()

In [0]:
#single file
struct1="cusid int, first_name string, last_name string, age int, profession string"
rawdf2=spark.read.schema(struct1).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified").show(truncate=False)
#multiple file
struct1="cusid int, first_name string, last_name string, age int, profession string"
rawdf2=spark.read.schema(struct1).csv(path=["/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified","/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified"]).show(truncate=False)



In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType
struct_type=StructType([StructField("cusid", IntegerType(),False), StructField("first_name", StringType(),False),StructField("last_name", StringType(),False), StructField("age", IntegerType(),False), StructField("profession", StringType(),False)])
rawdf2=spark.read.schema(struct_type).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one",recursiveFileLookup=True, pathGlobFilter="custsm*")
rawdf2.show(truncate=False)
rawdf2.printSchema()

### UnionByName 1st example

In [0]:
struct1="CUSID int, FIRST_NAME string, LAST_NAME string, AGE int, PROFESSION string"
raw_NY=spark.read.schema(struct1).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified_NY")
raw_NY.show(truncate=False)
struct2="CUSID int, FIRST_NAME string, AGE int, PROFESSION string, CITY string"
raw_TX=spark.read.schema(struct2).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified_TX")
raw_TX.show(truncate=False)
merged_NY_TX=raw_NY.unionByName(raw_TX,allowMissingColumns=True)
merged_NY_TX.show(truncate=False)


### Validation

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,ShortType
struct_type=StructType([StructField("cusid", IntegerType(),True), StructField("first_name", StringType(),True),StructField("last_name", StringType(),True), StructField("age", ShortType(),True), StructField("profession", StringType(),True),StructField("corrupt_record",StringType(),True)])
cleandf1=spark.read.schema(struct_type).csv(path="/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified",mode="PERMISSIVE") 
cleandf1.show(truncate=False)
cleandf1.count()

### Rejection strategy

In [0]:
from pyspark.sql.functions import col
rejecteddf=cleandf1.filter(col("corrupt_record").isNull())
rejecteddf.show(truncate=False)
print(len(rejecteddf.collect()))
rejecteddf=cleandf1.where("corrupt_record is not null")
print(len(rejecteddf.collect()))
display(rejecteddf)
#we can use both filter and where clause to filter out the corrupt records


### Cleansing using drop()

In [0]:
#df=df.na.drop(how="any") this function drops all the rows which has null values in any column, In spark every transformations prodices dataframe,
# transformations are applied on a Dataframe
str1="CUSID int, FIRST_NAME string, LAST_NAME string, AGE int, PROFESSION string"
cleanseddf1=spark.read.schema(str1).csv("/Volumes/catalog_one/schema_one/volume_one/directory_one/custsmodified")
cleanseddf2=cleanseddf1.na.drop(how="any")
display(cleanseddf1.where("age is  not null"))
print(cleanseddf1.count())
display(cleanseddf2.where("age is not null"))
print(cleanseddf2.count())

In [0]:
cleanseddf3=cleanseddf1.na.drop(how="any", subset=["cusid","age"])
display(cleanseddf3.where("cusid is not null"))
cleanseddf3=cleanseddf1.na.drop(how="all",subset=["cusid","age"])
display(cleanseddf3.where("cusid is not null"))

In [0]:
cleanseddf1=cleanseddf1.na.drop(how="all", subset=["first_name", "last_name"])
cleanseddf1.show()

### Scrubbing using fill() and replace()

In [0]:
scrubbeddf1=cleanseddf1.na.fill("not provided",subset=["last_name", "profession"])
scrubbeddf1.show(100,truncate=False)
replace_values1={"Actor":"Celebrity","Musician":"Composer"}
scrubbeddf2=scrubbeddf1.na.replace(replace_values1,subset=["profession"])
display(scrubbeddf2)


### Deduplication

In [0]:
#row-level
display(scrubbeddf2.where("cusid in ('4000001','4000003')"))
dedupdf1=scrubbeddf2.distinct() 
display(dedupdf1.coalesce(1).where("cusid in('4000001','4000003')")) 

#column-level
dedupdf2=dedupdf1.coalesce(1).dropDuplicates(subset=["cusid"])
display(dedupdf2.where("cusid = 4000003"))

#dedupdf2=dedupdf1.coalesce(1).where("cusid =4000003").orderBy(["cusid","age"],ascending=[True,False]).show()
dedupdf2=dedupdf1.coalesce(1).where("cusid =4000003").orderBy(["cusid","age"],ascending=[True,False]).dropDuplicates(subset=["cusid"])
display(dedupdf2.where("cusid = 4000003"))

### Standardization

#####Standardization1 - Column Enrichment (Addition of columns)

In [0]:
from pyspark.sql.functions import lit,initcap,col
standarddf1=scrubbeddf2.withColumn("sourcesystem",lit("retail"))
display(standarddf1.limit(20))

####Standardization2 - Column Uniformity

### withColumn

In [0]:
from pyspark.sql.functions import upper
display(dedupdf1.groupby("profession").count())
standarddf2=dedupdf1.withColumn("profession",initcap(col("profession")))
display(standarddf2)
standarddf3=standarddf2.withColumn("source system", lit("Retail"))
display(standarddf3)

###  Standardization3 - Format Standardization

In [0]:
from pyspark.sql.functions import lit,col,initcap,upper
standarddf3=rawdf1.where("id rlike '[a-z-|$]' ")
standarddf3.show()   #this is regualr expression like function
standarddf3=rawdf1.where("age  rlike '[^0-9]'")
standarddf3.show()
standarddf4=rawdf1.withColumn("Source System",lit("Retail")) #I did column enrichment here by adding a column since I used a different dataframe for column enrichment earlier
standarddf4.show(truncate=False)


In [0]:
from pyspark.sql.functions import *
replace_val={"one":'1',"two":'2',"three":'3',"four":'4',"five":'5',"six":'6',"seven":'7',"eight":'8',"nine":'9',"ten":'10'}
standarddf5= standarddf4.na.replace(replace_val,subset=["id"]) #applying to the values, completely changes it
standarddf6=standarddf5.withColumn("age",regexp_replace("age","-",""))#pattern based
display(standarddf6)

### Standardization4 - Data Type Standardization

In [0]:
from pyspark.sql.functions import col
standarddf6.printSchema()
standarddf7=standarddf6.withColumn("id",col("id").cast("long"))
standarddf8=standarddf7.withColumn("age",col("age").cast("int"))
standarddf8.printSchema()
standarddf8.show(100,False)



In [0]:
clean_df=standarddf6

In [0]:
standarddf9=clean_df.withColumn("id",expr("try_cast(id as bigint)")).filter(col("id").isNotNull())
display(standarddf9)

### Standardization5 - Naming Standardization

In [0]:
#standarddf9=standarddf8.withColumnRenamed("id","cus_id")
standarddf10=standarddf9.withColumnsRenamed({
  "id":"cus_id","source system":"src_system","fname":"first_name","lname":"last_name"
  })
display(standarddf10)

In [0]:
df=standarddf10.na.fill("not provided",subset=["last_name","profession"])
display(df)

### Standardization6 - Reorder Standadization

In [0]:
df=df.select("cus_id","age","first_name","last_name","profession","src_system")
mungeddf=df
display(mungeddf)

# 2. Data Enrichment - Detailing of data

#### a. Add (), Derive (), Rename (), Modify/replace (), Remove/Eliminate () - very important spark sql DF functions

#### Adding of columns

In [0]:
from pyspark.sql.functions import lit,current_date
derived_datadt="25/30/12"
print(f"Date is {derived_datadt}")
enricheddf1=mungeddf.withColumn("datadt",lit("25/30/12")).withColumn("loaddt",current_date())
#or
enricheddf1=mungeddf.withColumns({"datadt":lit("25/30/12"),"loaddt":current_date()})
#or
enricheddf1=mungeddf.select("*",lit(derived_datadt).alias("datadt"),current_date().alias("loaddt"))
display(enricheddf1)

In [0]:
enrich_df1=enricheddf1

#### Deriving of columns

In [0]:
from pyspark.sql.functions import *
enrichdf2=enrich_df1.withColumn("professionflag",substring("profession",1,1))
#or 
enrichdf2=enrich_df1.select("*",substring("profession",1,1).alias("professionflag"))
#or
enrichdf2=enrich_df1.selectExpr("*","substr(profession,1,1) as Professionflag")
display(enrichdf2)

#### Renaming of columns

In [0]:
enrichdf3=enrichdf2.withColumn("sourcename",col("src_system")) #it is costly, it will replicate the existing column data in a new column with new col name
enrichdf3=enrichdf3.drop("src_system").select("cus_id", "age","first_name","last_name","profession","sourcename","datadt","loaddt","Professionflag")
#or
enrichdf3=enrichdf2.select("cus_id","age","first_name","last_name","profession",col("src_system").alias("sourcename"),"datadt","loaddt","Professionflag")#costly too because we have select all columns
#or 
enrichdf3=enrichdf2.selectExpr("cus_id","age","first_name","last_name","profession","src_system as sourcename","datadt","loaddt","Professionflag") #similar to select
#or
enrichdf3=enrichdf2.withColumnRenamed("src_system","sourcename")
#or
enrichdf3=enrichdf2.withColumnsRenamed({"src_system":"sourcename","Professionflag":"profflag"})#similar to withColumn for multiple columns update
display(enrichdf3.take(50))


#### Modify/replace (withColumn, select/selectExpr)

In [0]:
enrichdf4=enrichdf3.withColumn("profession",col("sourcename"))#will replace profession with sourcename
#or
enrichdf4=enrichdf3.withColumn("profession",concat("profession",lit("_"),"profflag"))#This will overwrite the profession column witn sourcename
#or using select/selectExpr
enrichdf4=enrichdf3.select("cus_id","age","first_name","last_name",concat("profession",lit("_"),"profflag").alias("profession"),"sourcename","datadt","loaddt","profflag")
#selectExpr
enrichdf4=enrichdf3.selectExpr("cus_id","age","first_name","last_name","concat(profession, '_', profflag) as profession","sourcename","datadt","loaddt","profflag")
display(enrichdf4)


#### Remove/Eliminate (drop,select,selectExpr)

In [0]:
#withColumn cannot be use only select and selectExpr can be used but it is costly as we need to list all the columns
#select
enrichdf5=enrichdf4.select("cus_id","age","first_name","last_name","profession","sourcename","datadt","loaddt") #costly, here we unselect the column not drop
#selectExpr
enrichdf5=enrichdf4.selectExpr("cus_id","age","first_name","last_name","profession","sourcename","datadt","loaddt") #costly, here we unselect the column not drop
#or
enrichdf5=enrichdf4.drop("profflag")#correct method
display(enrichdf5)

In [0]:
##how to write a python program to append a variable value to another variable and use it inside the selectExpr
name='Ramesh'
sqlexpression=f"'{name}' as owner"
print(sqlexpression)
mungeddf.selectExpr("*",sqlexpression).display()

### Conclusion/Best practices of using different column enrichment functions
1.**select** is good to use if we want to perform - Good for ordering/reordering of columns, only renaming column (not good), only reformatting/deriving a column (not good), for all of these operation in a single iteration such renaming, reordering, reformatting,deriving, dropping etc., (best to use)

2.**selectExpr** is good to use if we want to perform - Same as select by using ISO/ANSI SQL functionality (if we are not familiar in DSL FBP) for all of these operation in a single iteration

3.**withColumn** is good to use if we want to perform - for adding/deriving/modifying/replacing in a single iteration Adding/Deriving column(s) in the last (Good), Modifying/replacing (Good), Renaming (not good), Dropping(not possible), reordering(not good)

4.**withColumnRenamed** is good to use if we want to perform - only for renaming column (Good)

5.**drop** is good to use if we want to perform - only dropping of columns in the given position (Good)`

#### b. Splitting & Merging/Melting of Columns

In [0]:
# Here we've splitted values from profession column into two indexes and applied it to newly created profflag column, taking only one index and applying back to profession column
splitdf=enrichdf5.withColumn("profflag",split("profession","_"))
splitdf=splitdf.withColumn("profession",col("profflag")[0]) #or [0] you can use
#or 
splitdf=splitdf.withColumn("shortprof",upper(substring(col("profession"),1,3))).drop("profflag")
#merging 
mergedf=splitdf.select("cus_id","age",concat_ws(" ",col("first_name"),col("last_name")).alias("full_name"),"profession","sourcename","datadt","loaddt","shortprof")
display(mergedf)

In [0]:
mergedf.printSchema()

#### c. Formatting & Typecasting

In [0]:
formatdf=mergedf.withColumn("datadt", to_date(col("datadt"),"yy/dd/MM"))
formatdf.printSchema()
display(formatdf)

In [0]:
def upper(colname):
    covertedcolvalue=colname.upper()
    return covertedcolvalue
print(upper("ramesh"))

In [0]:
from pyspark.sql.functions import udf
upperudf=udf(upper)
formatdf2=formatdf.withColumn("sourcename",upperudf(col("sourcename")))
display(formatdf2)

In [0]:
def pythonAgecat(dfcol):
    if dfcol is None:
        return "unknown"
    elif dfcol <=10:
        return "child"
    elif dfcol >10 and dfcol <=18:
        return "teenager"
    elif dfcol >18 and dfcol <=30:
        return "young"
    elif dfcol >30 and dfcol <=50:
        return "middleaged"
    else:
        return "senior"

In [0]:
formatdf2.printSchema()

In [0]:
typedf=formatdf2.withColumn("age",col("age").cast("int"))

In [0]:
from pyspark.sql.functions import udf
sparkudf=udf(pythonAgecat)
customdf=typedf.withColumn("agecat",sparkudf(col("age")))
display(customdf)